In [0]:
from pyspark.sql.functions import col, dayofweek, dayname, desc, sum, round

In [0]:
#path to valid sales table in silver schema
input_path = "incremental_sales.silver.sales"

In [0]:
df = spark.read.table(input_path)


In [0]:
# How much revenue was generated each day of the week?
weekly_df = (df
    .withColumn('day_name', dayname(col('invoice_date')))
    .withColumn('day_num', dayofweek(col('invoice_date')))
    .groupBy('day_name', 'day_num')
    .agg(round(sum(col('quantity') * col('unit_price')),2).alias('weekly_sales'))

    .write
    .format("delta")
    .mode('overwrite')
    .saveAsTable('incremental_sales.gold.weekly_sales')    
)

In [0]:
# Which 5 products generate the highest revenue?
top5_products_df = (df
    .groupBy('stock_code')
    .agg(round(sum(col('quantity') * col('unit_price')),2).alias('total_sales'))
    .orderBy(desc('total_sales'))
    .limit(5)

    .write
    .format("delta")
    .mode('overwrite')
    .saveAsTable('incremental_sales.gold.top5_products')    
)

In [0]:
# Which 5 customers generate the highest revenue?
top5_customers_df = (df
    .groupBy('customer_id')
    .agg(round(sum(col('quantity') * col('unit_price')),2).alias('total_sales'))
    .orderBy(desc('total_sales'))
    .limit(5)

    .write
    .format("delta")
    .mode('overwrite')
    .saveAsTable('incremental_sales.gold.top5_customers')    
)

In [0]:
# How much revenue was generated by each country?
country_revenue_df = (df
    .groupBy('country')
    .agg(round(sum(col('quantity') * col('unit_price')),2).alias('total_revenue'))

    .write
    .format("delta")
    .mode('overwrite')
    .saveAsTable('incremental_sales.gold.country_revenue')    
)

In [0]:
%sql
select * from incremental_sales.gold.weekly_sales limit 5

day_name,day_num,weekly_sales
Wed,4,120823.55
Fri,6,77618.55
Mon,2,78480.28
Sun,1,56494.76
Tue,3,99069.27
